# Pydantic + Spark Contract Pattern

This notebook shows how to combine **Pydantic** and **Apache Spark** as a clean engineering pattern.

The key idea:

```text
Pydantic = contract, documentation, edge validation
Spark    = distributed schema enforcement, validation, transformation, storage
```

This is **not** about validating every Spark row with Pydantic. That would usually be slow and unnecessary.

## Step 1 — Define the business contract with Pydantic

The Pydantic model describes the expected record shape. It is useful for API inputs, fixtures, tests, documentation, and small samples.

In [1]:
from datetime import datetime
from typing import Optional

from pydantic import BaseModel, Field


class CustomerEvent(BaseModel):
    customer_id: str = Field(min_length=1)
    event_type: str = Field(min_length=1)
    amount: Optional[float] = Field(default=None, ge=0)
    event_time: datetime

## Step 2 — Define the Spark schema explicitly

Keep the Spark schema explicit and readable. This avoids inference surprises and makes the ingestion contract obvious.

In [2]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
)


customer_event_schema = StructType([
    StructField("customer_id", StringType(), nullable=False),
    StructField("event_type", StringType(), nullable=False),
    StructField("amount", DoubleType(), nullable=True),
    StructField("event_time", TimestampType(), nullable=False),
])

customer_event_schema.simpleString()

'struct<customer_id:string,event_type:string,amount:double,event_time:timestamp>'

## Step 3 — Start Spark

This assumes the notebook runs inside your local Spark/Jupyter container. If `spark` already exists, this cell reuses it.

In [3]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder
        .appName("pydantic-spark-contract-pattern")
        .getOrCreate()
    )

spark.version

'4.0.2'

## Step 4 — Validate small samples at the edge

Use Pydantic before data becomes a distributed Spark workload. This is good for tests, API payloads, and small fixture files.

In [5]:
valid_payload = {
    "customer_id": "C001",
    "event_type": "purchase",
    "amount": 42.50,
    "event_time": "2026-05-02T10:00:00",
}

validated = CustomerEvent.model_validate(valid_payload)
validated

CustomerEvent(customer_id='C001', event_type='purchase', amount=42.5, event_time=datetime.datetime(2026, 5, 2, 10, 0))

In [7]:
invalid_payload = {
    "customer_id": "",
    "event_type": "purchase",
    "amount": -5,
    "event_time": "not-a-date",
}

try:
    CustomerEvent.model_validate(invalid_payload)
except Exception as ex:
    print(type(ex).__name__)
    print(ex)

ValidationError
3 validation errors for CustomerEvent
customer_id
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
amount
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal
event_time
  Input should be a valid datetime or date, invalid character in year [type=datetime_from_date_parsing, input_value='not-a-date', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/datetime_from_date_parsing


## Step 5 — Create a Spark DataFrame with the explicit schema

In a real pipeline this would be a `spark.read.schema(...).json(...)`, `csv(...)`, or streaming read.

In [9]:
from datetime import datetime

raw_customer_event_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", TimestampType(), True),
])

df = spark.createDataFrame(rows, schema=raw_customer_event_schema)
df.show(truncate=False)
df.printSchema()

+-----------+----------+------+-------------------+
|customer_id|event_type|amount|event_time         |
+-----------+----------+------+-------------------+
|C001       |purchase  |42.5  |2026-05-02 10:00:00|
|C002       |view      |NULL  |2026-05-02 10:01:00|
|           |purchase  |15.0  |2026-05-02 10:02:00|
|C004       |refund    |-7.0  |2026-05-02 10:03:00|
|C005       |NULL      |99.0  |2026-05-02 10:04:00|
+-----------+----------+------+-------------------+

root
 |-- customer_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



## Step 6 — Validate at scale with Spark expressions

This is where Spark should do the work. Avoid Pydantic UDFs for row-by-row validation.

In [10]:
from pyspark.sql import functions as F

valid_condition = (
    F.col("customer_id").isNotNull()
    & (F.length(F.trim(F.col("customer_id"))) > 0)
    & F.col("event_type").isNotNull()
    & (F.length(F.trim(F.col("event_type"))) > 0)
    & F.col("event_time").isNotNull()
    & (F.col("amount").isNull() | (F.col("amount") >= 0))
)

valid_df = df.filter(valid_condition)
quarantine_df = df.filter(~valid_condition)

valid_df.show(truncate=False)
quarantine_df.show(truncate=False)

+-----------+----------+------+-------------------+
|customer_id|event_type|amount|event_time         |
+-----------+----------+------+-------------------+
|C001       |purchase  |42.5  |2026-05-02 10:00:00|
|C002       |view      |NULL  |2026-05-02 10:01:00|
+-----------+----------+------+-------------------+

+-----------+----------+------+-------------------+
|customer_id|event_type|amount|event_time         |
+-----------+----------+------+-------------------+
|           |purchase  |15.0  |2026-05-02 10:02:00|
|C004       |refund    |-7.0  |2026-05-02 10:03:00|
|C005       |NULL      |99.0  |2026-05-02 10:04:00|
+-----------+----------+------+-------------------+



## Step 7 — Add quarantine reasons

In real pipelines, a quarantine dataset should explain why rows failed validation.

In [12]:
quarantine_with_reason_df = (
    df
    .withColumn(
        "validation_errors",
        F.array_remove(
            F.array(
                F.when(
                    F.col("customer_id").isNull() | (F.length(F.trim(F.col("customer_id"))) == 0),
                    F.lit("customer_id is required"),
                ),
                F.when(
                    F.col("event_type").isNull() | (F.length(F.trim(F.col("event_type"))) == 0),
                    F.lit("event_type is required"),
                ),
                F.when(
                    F.col("event_time").isNull(),
                    F.lit("event_time is required"),
                ),
                F.when(
                    F.col("amount") < 0,
                    F.lit("amount must be greater than or equal to zero"),
                ),
            ),
            None,
        )
    )
    .filter(F.size(F.col("validation_errors")) > 0)
)

quarantine_with_reason_df.show(truncate=False)

+-----------+----------+------+----------+-----------------+
|customer_id|event_type|amount|event_time|validation_errors|
+-----------+----------+------+----------+-----------------+
+-----------+----------+------+----------+-----------------+



## Step 8 — Optional write layout

Use separate outputs for valid and quarantined data.

Example folder layout:

```text
data/patterns/pydantic_spark/valid/
data/patterns/pydantic_spark/quarantine/
```

In [13]:
output_base = "/workspace/data/patterns/pydantic_spark"

# Uncomment when running inside the project container.
# valid_df.write.mode("overwrite").parquet(f"{output_base}/valid")
# quarantine_with_reason_df.write.mode("overwrite").parquet(f"{output_base}/quarantine")

## Summary

Use this pattern when you want clear contracts without slowing down Spark workloads.

```text
Good:
- Pydantic for contracts, configs, tests, and edge validation
- Spark StructType for distributed schema enforcement
- Spark expressions for large-scale validation
- quarantine tables for bad records

Avoid:
- Pydantic validation inside Spark UDFs for every row
- schema inference in serious ingestion pipelines
- mixing business contract logic only inside notebook cells with no reusable model
```